# Questão 5 - Sugestão de Arquivos

Suponha que vocês estão implementando uma solução que poderá ser utilizada por qualquer tipo de aplicação para recomendar qual pasta do computador possui mais arquivos relacionados.

### Exemplos

- Em uma aplicação de edição de imagens, a solução deve sugerir a pasta com mais imagens.
- Em uma aplicação para reproduzir músicas, a solução deve sugerir a pasta com mais arquivos de áudio.

### Estrutura de arquivos de exemplo

```text
home
├── user
│   ├── documentos
│   │   ├── relatorio.pdf
│   │   └── tcc.docx
│   ├── imagens
│   │   ├── foto1.jpg
│   │   └── foto2.jpg
│   └── musicas
│       └── rock.mp3
└── admin
    └── configs
        └── system.cfg
```

### O que deve ser feito

- Identificar e implementar qual estrutura melhor representa o problema.
- Implementar uma maneira de recomendar uma pasta, exibindo o path completo da pasta que possui o maior número de arquivos relacionados dado um tipo de arquivo informado pelo usuário.
- Exibir a listagem de arquivos e o caminho que foi percorrido para chegar nesses arquivos.


## Resolução

Para resolver este problema, utilizamos uma estrutura em **árvore** representada com dicionários e listas.

Cada pasta é um nó da árvore. Dentro de cada nó temos:
- o nome da pasta,
- a lista de arquivos dessa pasta,
- e a lista de subpastas.

Assim conseguimos percorrer toda a estrutura e descobrir qual pasta contém mais arquivos da extensão informada.

#### 1. Estrutura que representa o sistema de arquivos

Abaixo, montamos uma árvore de pastas de exemplo.

Cada pasta possui os campos `nome`, `arquivos` e `pastas` (filhos).

In [21]:
file_system = {
    "nome": "home",
    "arquivos": [],
    "pastas": [
        {
            "nome": "user",
            "arquivos": [],
            "pastas": [
                {
                    "nome": "documentos",
                    "arquivos": ["relatorio.pdf", "tcc.docx"],
                    "pastas": []
                },
                {
                    "nome": "imagens",
                    "arquivos": ["foto1.jpg", "foto2.jpg"],
                    "pastas": []
                },
                {
                    "nome": "musicas",
                    "arquivos": ["rock.mp3"],
                    "pastas": []
                },
            ]
        },
        {
            "nome": "admin",
            "arquivos": [],
            "pastas": [
                {
                    "nome": "configs",
                    "arquivos": ["system.cfg"],
                    "pastas": []
                }
            ]
        }
    ]
}

#### 2. Percurso em profundidade para encontrar a melhor pasta

Usaremos **DFS recursivo (pré-ordem)**.

Em cada pasta:
- montamos o caminho completo,
- filtramos os arquivos com a extensão desejada,
- comparamos com os resultados das subpastas.

Além disso, guardamos as pastas visitadas para exibir o caminho percorrido.

In [ ]:
def percorrer_pastas(pasta, extensao, caminho=""):
    novo_caminho = f"{caminho}/{pasta['nome']}" if caminho else pasta['nome']

    arquivos_relacionados = [
        arquivo for arquivo in pasta["arquivos"] if arquivo.lower().endswith(extensao)
    ]

    melhor_resultado = {
        "caminho": novo_caminho,
        "arquivos": arquivos_relacionados,
        "quantidade": len(arquivos_relacionados),
    }

    caminhos_visitados = [novo_caminho]

    for subpasta in pasta["pastas"]:
        resultado_subpasta = percorrer_pastas(subpasta, extensao, novo_caminho)
        caminhos_visitados.extend(resultado_subpasta["visitadas"])

        if resultado_subpasta["quantidade"] > melhor_resultado["quantidade"]:
            melhor_resultado = {
                "caminho": resultado_subpasta["caminho"],
                "arquivos": resultado_subpasta["arquivos"],
                "quantidade": resultado_subpasta["quantidade"],
            }

    melhor_resultado["visitadas"] = caminhos_visitados
    return melhor_resultado

#### 3. Recomendação e exibição dos resultados

Criamos uma função para:
- normalizar a extensão informada,
- chamar o percurso,
- exibir a pasta recomendada, a lista de arquivos e o caminho visitado.

No final, executamos com um exemplo.

In [ ]:
def normalizar_extensao(extensao):
    extensao = extensao.lower().strip()
    return extensao if extensao.startswith(".") else "." + extensao


def recomendar_pasta(estrutura, extensao):
    extensao = normalizar_extensao(extensao)
    resultado = percorrer_pastas(estrutura, extensao)

    print("Extensão informada:", extensao)
    print("Pasta recomendada:", resultado["caminho"])
    print("Quantidade de arquivos relacionados:", resultado["quantidade"])

    print("\nListagem de arquivos relacionados:")
    if resultado["arquivos"]:
        for arquivo in resultado["arquivos"]:
            print("-", f"{resultado['caminho']}/{arquivo}")
    else:
        print("Nenhum arquivo encontrado para essa extensão.")

    print("\nCaminho percorrido:")
    for caminho in resultado["visitadas"]:
        print("->", caminho)

    return resultado


recomendar_pasta(file_system, "jpg")

Pasta recomendada: home/user/imagens
Quantidade: 2
Arquivos: ['foto1.jpg', 'foto2.jpg']
Caminho percorrido:
-> home
-> user
-> imagens


## Complexidade de algoritmos


Análise do algoritmo implementado:

- **Tempo**: **O(p + a)**, onde `p` é o número de pastas e `a` é o número de arquivos. Isso ocorre porque cada pasta é visitada uma vez e cada arquivo é verificado no máximo uma vez.

- **Espaço**: **O(p)** para armazenar as pastas visitadas e, no pior caso, **O(h)** de pilha de recursão (`h` = altura da árvore). Assim, o espaço total fica **O(p + h)**.

A solução é linear em relação ao tamanho da estrutura percorrida.